## 工具调用的方式
###  **方式 1：** 直接调用

In [1]:
from dataclasses import Field

import pydantic
from langchain_core.tools import tool
from scripts.regsetup import description


@tool
def get_weather(city: str) -> str:
      """
      获取指定城市的天气信息

      参数 :
      city: 城市名称，如 " 北京 " 、 " 上海 "

      返回 :
      天气信息字符串
      """
      # 你的实现
      return city + " 晴天，温度 15°C"


# 使用 .invoke() 方法
result = get_weather.invoke({"city": " 北京 "})
print(result)

 北京  晴天，温度 15°C


### **方式 2：** 绑定到模型（主流）

In [37]:

import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        # model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_API_BASE)

In [6]:
from langchain_core.tools import tool


# 定义工具
@tool
def get_weather(city: str) -> str:
      """ 获取指定城市的天气 """
      # 你的实现
      return " 晴天，温度 15°C"


# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# AI 可以决定是否调用工具
response = model_with_tools.invoke(" 北京天气如何？ ")
# response = model_with_tools.invoke("2 + 3 = ？ ")

# 检查 AI 是否要调用工具
if response.tool_calls:
      print("AI 想调用工具： ", response.tool_calls)
else:
      print("AI 直接回答： ", response.content)

AI 想调用工具：  [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_EglaBpFpUnrwpz4kAYTs3389', 'type': 'tool_call'}]


##  2 工具调用的整体流程
### 从 Message 流转看工具的调用

In [18]:
from langchain.messages import HumanMessage, ToolMessage
from rich import print as rprint


def get_weather(city: str):
      """ 获取天气的工具 """

      return f"{city} 天气晴朗 "


# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

messages = [
      HumanMessage(" 今天北京天气如何 ")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加 AIMessage
messages.append(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
      if tool_call["name"] == "get_weather":
            # 拼接出 ToolMessage 实例
            tool_response = ToolMessage(
                  content=get_weather(**tool_call["args"]),
                  tool_call_id=tool_call["id"],
                  name=tool_call["name"]
            )
messages.append(tool_response)
rprint(tool_calls)
# print("=====================> messages <=====================")
# for msg in messages:
#       msg.pretty_print()
# print("=====================> messages <=====================")
# final_response = model_with_tools.invoke(messages)
# print(f"final_response: \n{final_response}")

[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_6KHtGW4uxGSVrxv0YAK97248', 'type': 'tool_call'}]

### 参考 2 ：使用 @tool 修饰

In [ ]:
from langchain.messages import HumanMessage, ToolMessage


@tool
def get_weather(city: str):
      """ 获取天气的工具 """
      return f"{city} 天气晴朗 ~"


# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

messages = [
      HumanMessage(" 今天北京天气如何 ")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加 AIMessage
messages.append(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
      if tool_call["name"] == "get_weather":
            # 返回的是 ToolMessage 类型消息
            tool_response = get_weather.invoke(tool_call)
            print(type(tool_response))
            messages.append(tool_response)

print("=====================> messages <=====================")
for msg in messages:
      msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

##  工具的定义方式 1 ：不使用 @tool
###  模型绑定工具并发送请求

In [ ]:
from rich import print as rprint


# 定义工具
def get_weather(city: str):
      return f"{city} 天气晴朗 "


# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(
      " 今天北京天气如何 "
)

rprint(response)
# print(response.tool_calls)

### 了解： convert_to_openai_tool

In [21]:
from langchain_core.utils.function_calling import convert_to_openai_tool

from rich import print as rprint


def get_weather(city: str):
      return f"{city} 天气晴朗 "


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

###  description 说明

In [22]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


def get_weather(city: str):
      """
      天气查询工具
      """
      return f"{city} 天气晴朗 "


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 参数默认值说明

In [23]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


def get_weather(city: str = " 北京 "):
      """
      天气查询工具

      Args:
      city: 城市名称
      """
      return f"{city} 天气晴朗 "


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {'city': {'default': ' 北京 ', 'description': '城市名称', 'type': 'string'}},
            'type': 'object'
        }
    }
}

## 工具的定义方式 2 ：使用 @tool 装饰器 ( 推荐 )
### 情况 1 ：仅提供 docstring 信息

In [30]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool


@tool
def get_weather(city: str):
      """ 天气预报工具 """
      return f"{city} 天气晴朗 "


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气预报工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 情况 2 ：添加工具描述： @tool description

In [32]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint


@tool(description=" 根据城市名称查询当日天气的工具 ")
def get_weather(city: str):
      """
      天气查询工具
      """
      return f"{city} 天气晴朗 "


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 情况 3 ：解析 docstring ： parse_docstring

In [56]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool(parse_docstring=True)
def get_weather(city: str, units: str = "celsius", include_forecast: bool = False) -> str:
      """
      获取当日天气，可选择是否同时查询未来五日天气预报

      Args:
          city: 城市
      units: 气温单位，可选： celsius- 摄氏度， fahrenheit- 华氏度
      include_forecast: 是否包含未来五日的天气预报
      """

      temp = 22 if units == "celsius" else 72

      result = f'{city} 当天气温 : {temp} {" 摄氏度 " if units == "celsius" else " 华氏度 "}'

      if include_forecast:
            result += "\n 未来五天都是晴天 "
            return result


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选： celsius- 摄氏度， fahrenheit- 华氏度',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

对比

In [55]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city: str, units: str = "celsius", include_forecast: bool =
False) -> str:
      """
      获取当日天气，可选择是否同时查询未来五日天气预报
      Args:
      city: 城市
      units: 气温单位，可选： celsius- 摄氏度， fahrenheit- 华氏度
      include_forecast: 是否包含未来五日的天气预报
      """
      temp = 22 if units == "celsius" else 72
      result = f'{city} 当天气温 : {temp} {" 摄氏度 " if units == "celsius" else " 华氏度 "}'
      if include_forecast:
            result += "\n 未来五天都是晴天 "
      return result


convert_to_openai_tool(get_weather)

{'type': 'function',
 'function': {'name': 'get_weather',
  'description': '获取当日天气，可选择是否同时查询未来五日天气预报\nArgs:\ncity: 城市\nunits: 气温单位，可选： celsius- 摄氏度， fahrenheit- 华氏度\ninclude_forecast: 是否包含未来五日的天气预报',
  'parameters': {'properties': {'city': {'type': 'string'},
    'units': {'default': 'celsius', 'type': 'string'},
    'include_forecast': {'default': False, 'type': 'boolean'}},
   'required': ['city'],
   'type': 'object'}}}

3.2 更改工具名称： name_or_callable

In [57]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool


# @tool 中参数 name_or_callable 名称可以省略
@tool("getWeather")
def get_weather(city: str):
      """
      天气查询工具
      """
      return f"{city} 天气晴朗 "


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 3自定义 args_schema
pydantic 类型的定义

In [61]:
from pydantic import BaseModel


class WeatherInput(BaseModel):
      city: str


@tool(args_schema=WeatherInput)
def get_weather(city: str):
      """ 天气预报工具 """

      return f"{city} 天气晴朗 "


rprint(convert_to_openai_tool(get_weather))


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气预报工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2

In [65]:
from pydantic import BaseModel
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
      city: str = Field(
            description="具体的城市",
            default="北京"
      )


@tool(args_schema=WeatherInput)
def get_weather(city: str):
      """ 天气预报工具 """

      return f"{city} 天气晴朗 "


rprint(convert_to_openai_tool(get_weather))


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气预报工具',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

In [70]:
from typing import Literal


class WeatherInput(BaseModel):
      city: str = Field(
            description="具体的城市",
            default="北京"
      )
      unit: Literal["celsius", "fahrenheit"]
      include_forecast: bool = Field(
            default=False,
            description="是否包含未来五天的天气预报"
      )


@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = False):
      """ 天气预报工具 """

      return f"{city} 天气晴朗 "


rprint(convert_to_openai_tool(get_weather))


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气预报工具',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

###   使用 Pydantic 定义 args_schema

In [75]:
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool


class WeatherInput(BaseModel):
      city: str = Field(
            default=" 北京 ",
            description=" 城市 "
      )

      unit: Literal["celsius", "fahrenheit"] = Field(
            default="celsius",
            description=" 气温单位 "
      )
      include_forecast: bool = Field(
            default=False,
            description=" 是否包含未来五日天气预报 "
      )


@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = False) -> str:
      """ 获取当日天气，可选未来五日天气预报 """

      temp = 22 if unit == "celsius" else 72
      result = f'{city} 当天气温 : {temp} {" 摄氏度 " if unit == "celsius" else " 华氏 度 "}'
      if include_forecast:
            result += "\n 未来五天都是晴天 "
      return result


result1 = convert_to_openai_tool(get_weather)
rprint(result)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'default': ' 北京 ', 'description': ' 城市 ', 'type': 'string'},
                'unit': {
                    'default': 'celsius',
                    'description': ' 气温单位 ',
                    'enum': ['celsius', 'fahrenheit'],
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': ' 是否包含未来五日天气预报 ',
                    'type': 'boolean'
                }
            },
            'type': 'object'
        }
    }
}

### 2 ：使用 Json Schema 定义

In [78]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

weather_schema = {
      "type": "object",
      "properties": {
            "location": {"type": "string"},
            "units": {"type": "string"},
            "include_forecast": {"type": "boolean"}
      },
      "required": ["location", "units", "include_forecast"]
}


@tool(args_schema=weather_schema)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool =
False) -> str:
      """ 获取当日天气，可选未来五日天气预报 """

      temp = 22 if unit == "celsius" else 72
      result = f'{city} 当天气温 : {temp} {" 摄氏度 " if unit == "celsius" else " 华氏度 "}'
      if include_forecast:
            result += "\n 未来五天都是晴天 "
      return result


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选未来五日天气预报',
        'parameters': {
            'type': 'object',
            'properties': {
                'location': {'type': 'string'},
                'units': {'type': 'string'},
                'include_forecast': {'type': 'boolean'}
            },
            'required': ['location', 'units', 'include_forecast']
        }
    }
}

## 1 ：使用 args_schema

In [88]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      # model_provider="deepseek",  # 已在 model 字符串中隐含指定，可省略
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
)

# ---------------------------------------------------------------------------
# 2. 定义工具（Tool）
# ---------------------------------------------------------------------------
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool


# 定义工具的参数结构（Schema），用于向模型描述工具的输入参数
class WeatherSchema(BaseModel):
      # 城市名称，默认值为 "北京"
      city: str = Field(default=" 北京 ", description=" 城市名称 ")
      # 是否包含明日天气预报，默认 False
      if_forecast: bool = Field(default=False, description=" 是否包含明日天气预报")


# 使用 @tool 装饰器把普通函数注册为 LangChain 工具：
# - name：工具名，模型通过该名称调用
# - description：工具用途说明，帮助模型判断何时调用
# - args_schema：参数结构，指定输入参数的字段与描述
@tool(
      "get_weather_and_forecast",
      description=" 查询当日天气，可以包含明日天气预报",
      args_schema=WeatherSchema,
)
def get_weather(city: str, if_forecast: bool):
      # 模拟返回当天的天气结果
      res = f"{city} 今天天气不错 "

      # 如果需要预报，则追加明日天气信息
      if if_forecast:
            res += "\n 明天也不错 "
      return res


# 将 LangChain 工具转换为 OpenAI 兼容的工具描述（JSON Schema 格式），便于调试查看
print(convert_to_openai_tool(get_weather))

# 把工具绑定到模型上，使模型具备调用该工具的能力
model_with_tools = model.bind_tools([get_weather])

# ---------------------------------------------------------------------------
# 3. 第一轮调用：让模型决定是否调用工具
# ---------------------------------------------------------------------------
messages = [HumanMessage(" 今天杭州天气如何？明天呢？ ")]

# 调用模型；模型可能会返回 tool_calls（决定调用哪些工具及参数）
response = model_with_tools.invoke(messages)

# 把模型的回复加入消息历史
messages.append(response)

# 取出模型请求调用的工具列表
tool_calls = response.tool_calls
rprint(tool_calls)  # 注：rprint 未在代码中导入，需 from rich import print as rprint

# ---------------------------------------------------------------------------
# 4. 执行工具调用，并把工具结果回传给模型
# ---------------------------------------------------------------------------
for tool_call in tool_calls:
      # 只处理我们定义的工具；tool_call 中包含 name / args / id 等信息
      if tool_call["name"] == "get_weather_and_forecast":
            # 以 tool_call 作为输入调用工具，返回 ToolMessage（内含 tool_call_id）
            tool_msg = get_weather.invoke(tool_call)

# 把工具执行结果加入消息历史，供模型在下一轮使用
messages.append(tool_msg)

# 再次调用模型，让模型结合工具结果生成最终回答
final_response = model_with_tools.invoke(messages)
messages.append(final_response)

# ---------------------------------------------------------------------------
# 5. 打印完整的消息对话记录
# ---------------------------------------------------------------------------
for msg in messages:
      # pretty_print 会以人类可读的格式打印每条消息（角色、内容、工具调用等）
      msg.pretty_print()

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '查询当日天气，可以包含明日天气预报', 'parameters': {'properties': {'city': {'default': ' 北京 ', 'description': ' 城市名称 ', 'type': 'string'}, 'if_forecast': {'default': False, 'description': ' 是否包含明日天气预报', 'type': 'boolean'}}, 'type': 'object'}}}


[
    {
        'name': 'get_weather_and_forecast',
        'args': {'city': '杭州', 'if_forecast': True},
        'id': 'call_00_ET_dIW5rMwjNQSpmAVUFN0H2974',
        'type': 'tool_call'
    }
]

================================ Human Message =================================

 今天杭州天气如何？明天呢？ 
================================== Ai Message ==================================

我来帮你查询杭州的天气。
Tool Calls:
  get_weather_and_forecast (call_00_ET_dIW5rMwjNQSpmAVUFN0H2974)
 Call ID: call_00_ET_dIW5rMwjNQSpmAVUFN0H2974
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州 今天天气不错 
 明天也不错 
================================== Ai Message ==================================

查询结果如下：

**杭州**
- **今天**：天气不错
- **明天**：也不错

不过这次查询只返回了概括性的描述，没有具体的温度、湿度、降水概率等数据。如果你需要更精确的信息（比如具体气温、是否要带伞、穿衣建议），可以告诉我，我可以再帮你查一次，或者你也可以告诉我你关注哪些细节。


##  案例 2 ：撰写 docstring
可以在 docstring 中撰写参数的描述信息，此时参数默认值和类型都要通过函数签名传递。
模型初始化

In [90]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import BaseMessage

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
)
from langchain.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool


## 调用工具类
@tool("get_weather_and_forecast", parse_docstring=True)
def get_weather(city: str = " 北京 ", if_forecast: bool = False):
      """
      查询当日天气，可以包含明日天气预报

      Args:
      city: 城市名称
      if_forecast: 是否包含明日天气预报
      """
      res = f"{city} 今天天气不错 "
      if if_forecast:
            res += "\n 明天要下雨 "
      return res


# convert_to_openai_tool  把 LangChain 的工具对象转换成 OpenAI 函数调用
print(convert_to_openai_tool(get_weather))

model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage(" 今天杭州天气如何？明天呢？ ")]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_calls = response.tool_calls

# 将工具调用的结果添加到消息列表中
for tool_call in tool_calls:
      if tool_call["name"] == "get_weather_and_forecast":
            # 返回值 tool_msg 类型是 ToolMessage
            tool_msg = get_weather.invoke(tool_call)
            messages.append(tool_msg)

final_response = model_with_tools.invoke(messages)

messages.append(final_response)

for msg in messages:
      msg.pretty_print()

{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '查询当日天气，可以包含明日天气预报', 'parameters': {'properties': {'city': {'default': ' 北京 ', 'description': '城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明日天气预报', 'type': 'boolean'}}, 'type': 'object'}}}
================================ Human Message =================================

 今天杭州天气如何？明天呢？ 
================================== Ai Message ==================================

我来帮您查询杭州的天气情况。
Tool Calls:
  get_weather_and_forecast (call_00_KwnRps3iHxOO6J1BoocM7374)
 Call ID: call_00_KwnRps3iHxOO6J1BoocM7374
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

杭州 今天天气不错 
 明天要下雨 
================================== Ai Message ==================================

杭州的天气情况如下：

- **今天**：天气不错，适合外出活动 ☀️
- **明天**：有雨，建议出门记得带伞 🌧️

明天出门记得携带雨具，注意安全～


##  案例 3 ：多工具调用
大模型调用工具是单次推理，即每次运行调用一个工具，当调用多个工具时，需要用户自己管理多次调
用循环

In [100]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


# ==========================================
# 1. 定义工具
# ==========================================

# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
      """获取指定公司的股票价格信息

      Args:
          company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
          timeframe: 时间范围（today-今日, week-本周, month-本月）
      """
      # 模拟股票数据
      mock_data = {
            "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
            "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
            "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
      }

      if company in mock_data:
            price = mock_data[company].get(timeframe, "未知时间范围")
            return f"{company} {timeframe} 价格: {price} 美元"
      else:
            return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
      """搜索指定公司的财经新闻

      Args:
          company: 公司名称
      """
      # 模拟新闻数据
      mock_news = {
            "苹果公司": [
                  "苹果发布新款 iPhone，股价上涨 3%",
                  "苹果与欧盟达成反垄断和解协议",
                  "苹果将在印度扩大生产规模"
            ],
            "微软公司": [
                  "微软 Azure 云业务季度增长超预期",
                  "微软完成对 Nuance 的收购",
                  "微软推出新一代 AI 助手 Copilot"
            ],
            "谷歌公司": [
                  "谷歌发布新 AI 模型，性能提升 20%",
                  "谷歌与 OpenAI 合作，开发新的 AI 助手",
                  "谷歌在欧洲展开 AI 研究项目"
            ]
      }

      news_list = mock_news.get(company, [f"未找到 {company} 的相关新闻"])
      return "\n".join(news_list)


# ==========================================
# 2. 初始化模型并绑定工具
# ==========================================
# 注意：这里假设你的 model 已经在外部定义好了
# model = init_chat_model(...)

tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)

# 初始化对话历史
messages = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
messages.append(human_message)

# ==========================================
# 3. 工具调用循环 (Agent 核心逻辑)
# ==========================================
while True:
      # 1. 将当前对话历史发给大模型
      response = model_with_tools.invoke(messages)

      # 2. 将大模型的回复（可能包含 tool_calls）加入历史
      messages.append(response)

      # 3. 如果模型不需要调用工具，说明它已经得出最终答案了，退出循环
      if not response.tool_calls:
            print("\n=== 模型最终回答 ===")
            print(response.content)
            break

      # 4. 如果模型请求调用工具，则根据请求执行对应的工具
      print(f"\n[模型请求调用 {len(response.tool_calls)} 个工具]")
      # 如果有调用工具，处理工具调用响应
      # 4. 开发者根据模型的响应，调用工具并获取结果
      for tool_call in response.tool_calls:
            if tool_call["name"] == "get_stock_price":
                  stock_result = get_stock_price.invoke(tool_call)
                  print("stock_result", stock_result)
                  messages.append(stock_result)

            elif tool_call["name"] == "search_news":
                  news_result = search_news.invoke(tool_call)
                  print("news_result", news_result)
                  messages.append(news_result)

      # print("response", response)
      # print(response.content)

      for msg in messages:
            msg.pretty_print()




[模型请求调用 2 个工具]
stock_result content='苹果公司 today 价格: 185.2 美元' name='get_stock_price' tool_call_id='call_00_ET_nM5FQvik6l4jiLYQ2vSx1819'
news_result content='苹果发布新款 iPhone，股价上涨 3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模' name='search_news' tool_call_id='call_01_ET_o6ABM9HbMl2Ye8Ol6u8P2647'
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻？
================================== Ai Message ==================================

我来帮您查询苹果公司的股价和最新新闻。
Tool Calls:
  get_stock_price (call_00_ET_nM5FQvik6l4jiLYQ2vSx1819)
 Call ID: call_00_ET_nM5FQvik6l4jiLYQ2vSx1819
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_01_ET_o6ABM9HbMl2Ye8Ol6u8P2647)
 Call ID: call_01_ET_o6ABM9HbMl2Ye8Ol6u8P2647
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司 today 价格: 185.2 美元
================================= Tool Message =================================
Name: search_news



## 举例2

In [101]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
      """
      获取当日天气

      Args:
      city: 城市名称
      """
      return f'{city} 当天晴朗 '


@tool(parse_docstring=True)
def get_news() -> str:
      """
      获取当日新闻
      """
      return " 近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来 “ 火热潮 ” ，回收价格普遍上涨，旧手机成 “ 香饽饽 ” 。 "


model_with_tools = model.bind_tools([get_weather, get_news])

messages = [
      HumanMessage(" 今天杭州天气如何？今天新闻是什么？别瞎编 ")
]

response = model_with_tools.invoke(messages)
response.pretty_print()

# ==================================
#  Ai Message
# ==================================
# 我来帮您查询杭州的天气和今日新闻。
# Tool Calls:
#   get_weather (call_00_uspkbqR7N7wIewhr8hHZAXoM)
#   Call ID: call_00_uspkbqR7N7wIewhr8hHZAXoM
#   Args:
#     city: 杭州
#   get_news (call_01_ksyZoO2kXFBXXp1dH4dnxZAN)
#   Call ID: call_01_ksyZoO2kXFBXXp1dH4dnxZAN
#   Args:

# 此时可以遍历 tool_calls 列表，挨个调用工具。
messages.append(response)

for tool_call in response.tool_calls:
      if tool_call["name"] == "get_weather":
            tool_msg = get_weather.invoke(tool_call)
            print(tool_msg)
            messages.append(tool_msg)
      elif tool_call["name"] == "get_news":
            tool_msg = get_news.invoke(tool_call)
            print(tool_msg)
            messages.append(tool_msg)
      else:
            raise Exception(" 不存在的工具 ")

final_response = model.invoke(messages)
messages.append(final_response)

for msg in messages:
      msg.pretty_print()

================================== Ai Message ==================================

我来查询一下杭州的天气和今天的新闻。
Tool Calls:
  get_weather (call_00_Icg3xhfrkgxkPvp0wP3s3958)
 Call ID: call_00_Icg3xhfrkgxkPvp0wP3s3958
  Args:
    city: 杭州
  get_news (call_01_hDXhZMcC09Nu1JpLQ0tw9405)
 Call ID: call_01_hDXhZMcC09Nu1JpLQ0tw9405
  Args:
content='杭州 当天晴朗 ' name='get_weather' tool_call_id='call_00_Icg3xhfrkgxkPvp0wP3s3958'
content=' 近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来 “ 火热潮 ” ，回收价格普遍上涨，旧手机成 “ 香饽饽 ” 。 ' name='get_news' tool_call_id='call_01_hDXhZMcC09Nu1JpLQ0tw9405'
================================ Human Message =================================

 今天杭州天气如何？今天新闻是什么？别瞎编 
================================== Ai Message ==================================

我来查询一下杭州的天气和今天的新闻。
Tool Calls:
  get_weather (call_00_Icg3xhfrkgxkPvp0wP3s3958)
 Call ID: call_00_Icg3xhfrkgxkPvp0wP3s3958
  Args:
    city: 杭州
  get_news (call_01_hDXhZMcC09Nu1JpLQ0tw9405)
 Call ID: call_01_hDXhZMcC09Nu1JpLQ0tw9405
  Args:
=================